In [ ]:
paid = data["paid"]
orders = data["orders"]
customers = data["customers"]

n_total = customers["Customer Key"].nunique()
n_repeat = customers["Repeat_Buyer"].sum()
repeat_pct = n_repeat / n_total * 100 if n_total else 0
repeat_revenue_pct = customers.loc[customers["Repeat_Buyer"], "Total_Spend"].sum() / customers["Total_Spend"].sum() * 100

st.subheader("Customer Overview")
c1, c2, c3, c4 = st.columns(4)
with c1:
    kpi_card("Unique Customers", f"{n_total:,}")
with c2:
    kpi_card("Repeat Customers", f"{n_repeat:,.0f}")
with c3:
    kpi_card("Repeat Customer Rate", f"{repeat_pct:.2f}%")
with c4:
    kpi_card("Revenue from Repeat", f"{repeat_revenue_pct:.1f}%")

In [ ]:
# Monthly new vs repeat
monthly_nr = paid.groupby(["Paid Month", "Is New Customer Order"]).size().unstack(fill_value=0)
monthly_nr.columns = ["Repeat" if c is False else "New" for c in monthly_nr.columns]
monthly_nr = monthly_nr.reset_index()
monthly_nr["Paid Month"] = monthly_nr["Paid Month"].astype(str)

fig = go.Figure()
if "New" in monthly_nr.columns:
    fig.add_bar(x=monthly_nr["Paid Month"], y=monthly_nr["New"], name="New Customers", marker_color=SUCCESS)
if "Repeat" in monthly_nr.columns:
    fig.add_bar(x=monthly_nr["Paid Month"], y=monthly_nr["Repeat"], name="Repeat Customers", marker_color=ACCENT1)
fig.update_layout(barmode="stack", title="Monthly Orders: New vs Repeat Customers")
st.plotly_chart(fig, width='stretch')

In [ ]:
# Repeat Customer Seasonality
if "Order Seq" in paid.columns and "Paid at" in paid.columns:
    st.subheader("Repeat Customer Seasonality")
    st.caption(
        "Across the full order history, which calendar months see the most repeat-customer "
        "activity? Useful for timing loyalty campaigns, restocks, or win-back offers around "
        "your repeat-buyer peaks."
    )

    repeat_orders = paid[paid["Order Seq"] > 1].copy()
    repeat_orders["Calendar Month"] = repeat_orders["Paid at"].dt.month_name()

    month_order = ["January", "February", "March", "April", "May", "June",
                    "July", "August", "September", "October", "November", "December"]
    monthly_repeat_seasonal = (
        repeat_orders.groupby("Calendar Month").size()
        .reindex(month_order, fill_value=0)
        .reset_index()
    )
    monthly_repeat_seasonal.columns = ["Month", "Repeat Orders"]

    fig = px.bar(monthly_repeat_seasonal, x="Month", y="Repeat Orders",
                  title="Repeat Customer Orders by Calendar Month (All Years Combined)",
                  color_discrete_sequence=[ACCENT1])
    st.plotly_chart(fig, width='stretch')

    top_month = monthly_repeat_seasonal.loc[monthly_repeat_seasonal["Repeat Orders"].idxmax()]
    st.caption(
        f"**{top_month['Month']}** sees the most repeat-customer orders "
        f"({top_month['Repeat Orders']:,}) across the dataset's history -- a strong candidate "
        "month for loyalty/win-back campaigns."
    )

In [ ]:
col1, col2 = st.columns(2)
with col1:
    # Order frequency distribution
    dist = customers["Orders"].value_counts().sort_index().head(8).reset_index()
    dist.columns = ["Orders", "Customers"]
    dist["Orders"] = dist["Orders"].astype(str)
    fig = px.bar(dist, x="Orders", y="Customers", title="Customer Order Frequency Distribution",
                  color_discrete_sequence=[ACCENT2])
    st.plotly_chart(fig, width='stretch')

with col2:
    # Tier breakdown
    tier_table = customers.groupby("Tier", observed=False).agg(
        Customers=("Customer Key", "count"),
        Total_Revenue=("Total_Spend", "sum"),
    ).reset_index()
    fig = px.bar(tier_table, x="Tier", y="Total_Revenue", title="Revenue by Customer Tier",
                  color="Tier", color_discrete_sequence=PALETTE)
    st.plotly_chart(fig, width='stretch')

In [ ]:
st.subheader("Customer Tier Table")
tier_table_full = customers.groupby("Tier", observed=False).agg(
    Customers=("Customer Key", "count"),
    Avg_Spend=("Total_Spend", "mean"),
    Total_Revenue=("Total_Spend", "sum"),
    Avg_Orders=("Orders", "mean"),
    Repeat_Rate_Pct=("Repeat_Buyer", lambda x: x.mean() * 100),
).round(2)
tier_table_full["Revenue Share %"] = (tier_table_full["Total_Revenue"] / tier_table_full["Total_Revenue"].sum() * 100).round(2)
st.dataframe(tier_table_full, width='stretch')

In [ ]:
# 1st-to-2nd order gap
if "Order Seq" in paid.columns:
    st.subheader("1st-to-2nd Order Gap")
    first_two = paid[paid["Order Seq"].isin([1, 2])][["Customer Key", "Order Seq", "Paid at"]]
    pivot = first_two.pivot(index="Customer Key", columns="Order Seq", values="Paid at").dropna()
    if not pivot.empty and 1 in pivot.columns and 2 in pivot.columns:
        gap_days = (pivot[2] - pivot[1]).dt.total_seconds() / 86400
        c1, c2 = st.columns(2)
        with c1:
            kpi_card("Avg Gap (1st->2nd order)", f"{gap_days.mean():.1f} days")
        with c2:
            kpi_card("Median Gap", f"{gap_days.median():.1f} days")
        fig = px.histogram(gap_days, nbins=40, title="Distribution: Days Between 1st and 2nd Order",
                            color_discrete_sequence=[ACCENT1])
        fig.update_layout(showlegend=False, xaxis_title="Days", yaxis_title="Customers")
        st.plotly_chart(fig, width='stretch')

In [ ]:
# Top customers
st.subheader("Top Customers by Lifetime Spend")
top_cust = customers.sort_values("Total_Spend", ascending=False).head(15)[
    ["Customer Key", "Orders", "Total_Spend", "AOV", "Tier"]
]
st.dataframe(top_cust, width='stretch')

In [ ]:
# Customer Order Lookup (drill-down into a single customer's orders)
st.subheader("Customer Order Lookup")
st.caption(
    "Search for a customer (by name or zip -- `Customer Key` is `billing name | billing zip`) "
    "to see their full order history."
)

search = st.text_input("Search Customer Key", placeholder="e.g. ramesh kumar or 110092")
if search.strip():
    matches = customers[customers["Customer Key"].str.contains(search.strip().lower(), case=False, na=False)]
    matches = matches.sort_values("Total_Spend", ascending=False).head(50)
    if matches.empty:
        st.info("No customers found matching that search.")
    else:
        options = matches["Customer Key"].tolist()
        selected = st.selectbox("Matching customers", options, format_func=lambda k:
                                 f"{k}  ({int(matches.loc[matches['Customer Key'] == k, 'Orders'].iloc[0])} orders, "
                                 f"₹{matches.loc[matches['Customer Key'] == k, 'Total_Spend'].iloc[0]:,.0f})")

        cust_orders = orders[orders["Customer Key"] == selected].sort_values("Created at")
        display_cols = [c for c in [
            "Name", "Created at", "Paid at", "Financial Status", "Fulfillment Status",
            "Total", "Payment Type", "Any RTO", "Shipping City", "Tags",
        ] if c in cust_orders.columns]
        st.dataframe(cust_orders[display_cols], width="stretch")

In [ ]:
# Revenue Concentration (Pareto)
st.markdown("---")
st.subheader("Revenue Concentration")

pareto = customers.sort_values("Total_Spend", ascending=False).reset_index(drop=True)
pareto["Cum_Revenue_Pct"] = pareto["Total_Spend"].cumsum() / pareto["Total_Spend"].sum() * 100
pareto["Cum_Customer_Pct"] = (pareto.index + 1) / len(pareto) * 100

fig = go.Figure()
fig.add_scatter(x=pareto["Cum_Customer_Pct"], y=pareto["Cum_Revenue_Pct"], mode="lines",
                 line=dict(color=ACCENT1), fill="tozeroy", fillcolor="rgba(91,141,239,0.10)",
                 name="Cumulative Revenue")
fig.add_scatter(x=[0, 100], y=[0, 100], mode="lines", line=dict(color=PALETTE[4], dash="dash"),
                 name="Equal Distribution")
fig.update_layout(title="Cumulative Revenue Share vs Cumulative Customer Share",
                   xaxis_title="Customers (cumulative %)", yaxis_title="Revenue (cumulative %)")
st.plotly_chart(fig, width="stretch")

top10_pct = pareto.loc[pareto["Cum_Customer_Pct"] <= 10, "Total_Spend"].sum() / pareto["Total_Spend"].sum() * 100
top20_pct = pareto.loc[pareto["Cum_Customer_Pct"] <= 20, "Total_Spend"].sum() / pareto["Total_Spend"].sum() * 100
st.caption(
    f"The top 10% of customers by lifetime spend account for **{top10_pct:.1f}%** of total revenue, "
    f"and the top 20% account for **{top20_pct:.1f}%**. With such a low repeat-purchase rate, this "
    "concentration comes mostly from a small group of high-AOV one-time buyers rather than loyal "
    "repeat customers -- worth treating as a VIP/high-value segment for personalized outreach."
)

In [ ]:
# New Customer Acquisition Trend
st.subheader("New Customer Acquisition Trend")

acq = customers.copy()
acq["Cohort Month"] = acq["First_Order"].dt.to_period("M")
acq_trend = acq.groupby("Cohort Month").size().reset_index(name="New Customers")
acq_trend["Cohort Month"] = acq_trend["Cohort Month"].astype(str)

fig = px.bar(acq_trend, x="Cohort Month", y="New Customers", title="New Customers Acquired per Month",
              color_discrete_sequence=[SUCCESS])
st.plotly_chart(fig, width="stretch")
st.caption(
    "Counts customers by the month of their first paid order. The most recent month is often "
    "incomplete (data export cutoff)."
)

In [ ]:
# Customer Geography by State
if "Shipping Province" in paid.columns:
    st.subheader("Customer Geography")

    prov = paid.groupby("Customer Key")["Shipping Province"].first()
    cg = customers.merge(prov.rename("Shipping Province"), on="Customer Key", how="left")
    geo = cg.groupby("Shipping Province").agg(
        Customers=("Customer Key", "count"),
        Total_Revenue=("Total_Spend", "sum"),
        Avg_Spend=("Total_Spend", "mean"),
        Repeat_Rate_Pct=("Repeat_Buyer", lambda x: x.mean() * 100),
    ).round(2).sort_values("Customers", ascending=False).head(10)

    col1, col2 = st.columns(2)
    with col1:
        geo_bar = geo.reset_index()
        fig = px.bar(geo_bar, x="Customers", y="Shipping Province", orientation="h",
                      title="Top 10 States by Customer Count", color_discrete_sequence=[ACCENT1])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width="stretch")
    with col2:
        st.dataframe(geo, width="stretch")

In [ ]:
# Repeat Purchase Timing by Acquisition Cohort
if "Order Seq" in paid.columns:
    st.subheader("Repeat Purchase Timing by Acquisition Cohort")

    first_two = paid[paid["Order Seq"].isin([1, 2])][["Customer Key", "Order Seq", "Paid at"]]
    gap_pivot = first_two.pivot(index="Customer Key", columns="Order Seq", values="Paid at").dropna()
    if 1 in gap_pivot.columns and 2 in gap_pivot.columns:
        gap_days = (gap_pivot[2] - gap_pivot[1]).dt.days

        cohort_map = customers.set_index("Customer Key")["First_Order"].dt.to_period("M")
        repeat_df = pd.DataFrame({"Gap": gap_days})
        repeat_df["Cohort Month"] = repeat_df.index.map(cohort_map)
        repeat_30 = repeat_df.groupby("Cohort Month").apply(lambda g: (g["Gap"] <= 30).sum())

        cohort_sizes = customers.copy()
        cohort_sizes["Cohort Month"] = cohort_sizes["First_Order"].dt.to_period("M")
        cohort_totals = cohort_sizes.groupby("Cohort Month").size()

        rate30 = (repeat_30 / cohort_totals * 100).fillna(0).reset_index()
        rate30.columns = ["Cohort Month", "30-Day Repeat Rate %"]
        rate30 = rate30.iloc[:-1]  # drop most recent (incomplete) cohort
        rate30["Cohort Month"] = rate30["Cohort Month"].astype(str)

        fig = px.line(rate30, x="Cohort Month", y="30-Day Repeat Rate %", markers=True,
                       title="Share of Each Acquisition Cohort That Repeat-Purchased Within 30 Days")
        fig.update_traces(line_color=ACCENT2, fill="tozeroy", fillcolor="rgba(124,147,184,0.10)")
        st.plotly_chart(fig, width="stretch")
        st.caption(
            "For each monthly cohort of new customers, the share that placed a second paid order "
            "within 30 days of their first. The most recent (incomplete) cohort is excluded. "
            "Useful for spotting whether early-lifecycle retention efforts (e.g. post-purchase "
            "emails/offers) are improving over time."
        )